In [1]:
from pathlib import Path
import math
import random

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, box
from shapely.affinity import translate

In [2]:
from tgbs_rs.config.config import AOI_PATHS, OUTPUT_DIRS 

In [74]:
SRC_CRS = "EPSG:4326"
WORK_CRS = "EPSG:32737"
GRID_SIZE_M = 100
PLOT_SIZE_M = 20
PLOT_AREA_HA = 0.04
EDGE_BUFFER_M = 50
MIN_SEPARATION_M = 200
SAMPLING_PCT = 0.03  # change to 0.03 or 0.04 as needed
RANDOM_SEED = 42

blocks_path = AOI_PATHS["ks_rehab_blocks"]
corridor_path = AOI_PATHS["corridor"]
out_dir = OUTPUT_DIRS["sampling_design"]

In [75]:
def load_blocks_geojson(path: str | Path, work_crs: str = "EPSG:32737") -> gpd.GeoDataFrame:
    """
    Load the restoration blocks and reproject them to a projected CRS for area and distance calculations.
    Rows without a valid block ID or year are removed so only true restoration blocks remain and IDs are normalized as strings.
    """
    gdf = gpd.read_file(path)
    gdf = gdf.loc[gdf["id"].notna() & gdf["year"].notna()].copy()
    gdf["id"] = gdf["id"].astype(str).str.replace(r"\.0$", "", regex=True)
    gdf["year"] = gdf["year"].astype(int)
    gdf = gdf.to_crs(work_crs)
    return gdf


def assign_restoration_strata(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Assign stratum labels from the restoration start year for each block.
    Years 2020–2021 become Early Restoration and years 2022–2024 become Recent Restoration.
    """
    out = gdf.copy()
    out["Stratum"] = np.where(
        out["year"].between(2020, 2021),
        "Early Restoration",
        np.where(out["year"].between(2022, 2024), "Recent Restoration", "Unassigned"),
    )
    return out


def recalculate_block_area_ha(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Recalculate exact block area from geometry in a projected CRS and store it in hectares.
    This avoids relying on the source attribute field when proportional allocation is needed.
    """
    out = gdf.copy()
    out["area_m2_calc"] = out.geometry.area
    out["area_ha_calc"] = out["area_m2_calc"] / 10000.0
    return out


def load_corridor_geojson(path: str | Path, work_crs: str = "EPSG:32737") -> gpd.GeoDataFrame:
    """
    Load the biodiversity corridor polygons and prepare them to match the restoration block schema.
    Existing corridor IDs are preserved so the corridor remains a separate sampling unit in all outputs.
    """
    gdf = gpd.read_file(path).to_crs(work_crs).copy()
    gdf["id"] = gdf["id"].astype(str).str.replace(r"\.0$", "", regex=True)
    gdf["year"] = pd.NA
    gdf["Stratum"] = "Biodiversity Corridor"
    return gdf


def build_sampling_zone(gdf: gpd.GeoDataFrame, edge_buffer_m: float = 50) -> gpd.GeoDataFrame:
    """
    Create the valid plot-center zone by buffering each polygon inward from its boundary.
    """
    out = gdf.copy()
    out["orig_geometry"] = out.geometry
    out["geometry"] = out.geometry.buffer(-edge_buffer_m)
    out["sampling_ok"] = ~out.geometry.is_empty
    out = out.loc[out["sampling_ok"]].copy()
    out = out.set_geometry("geometry")
    return out

def compute_target_plot_count(gdf: gpd.GeoDataFrame, sampling_pct: float) -> gpd.GeoDataFrame:
    """
    Compute the literal target number of 20 x 20 m plots from a chosen percentage of block area.
    Plot counts are rounded up so the allocation never falls below the requested intensity.
    """
    out = gdf.copy()
    out["target_n"] = np.ceil((out["area_ha_calc"] * sampling_pct) / 0.04).astype(int)
    return out


def make_shifted_grid(bounds: tuple, cell_size: float, offset_x: float, offset_y: float) -> gpd.GeoDataFrame:
    """
    Build a square grid covering the input bounds using a randomized grid origin.
    The grid is returned as polygons so centroids can later be filtered within each sampling zone.
    """
    minx, miny, maxx, maxy = bounds
    start_x = math.floor((minx - offset_x) / cell_size) * cell_size + offset_x
    start_y = math.floor((miny - offset_y) / cell_size) * cell_size + offset_y

    xs = np.arange(start_x, maxx + cell_size, cell_size)
    ys = np.arange(start_y, maxy + cell_size, cell_size)

    cells = []
    for x in xs[:-1]:
        for y in ys[:-1]:
            cells.append(box(x, y, x + cell_size, y + cell_size))

    return gpd.GeoDataFrame(geometry=cells, crs=WORK_CRS)


def candidate_points_from_block(
    block_geom,
    grid_size_m: float = 100,
    seed: int = 42,
) -> tuple[gpd.GeoDataFrame, gpd.GeoDataFrame]:
    """
    Generate a systematic-random 100 m grid and the candidate plot centers for one block.
    The function returns both the shifted grid polygons and the centroid points that fall inside the valid sampling zone.
    """
    rng = random.Random(seed)
    offset_x = rng.uniform(0, grid_size_m)
    offset_y = rng.uniform(0, grid_size_m)

    grid = make_shifted_grid(block_geom.bounds, grid_size_m, offset_x, offset_y)
    grid = grid.copy()
    grid["grid_id"] = [f"G{i+1:04d}" for i in range(len(grid))]
    grid["offset_x"] = offset_x
    grid["offset_y"] = offset_y

    candidates = grid.copy()
    candidates["geometry"] = candidates.geometry.centroid
    candidates = candidates.loc[candidates.within(block_geom)].copy()

    return grid, candidates


def select_points_with_min_spacing(points_gdf: gpd.GeoDataFrame, target_n: int, min_sep_m: float, seed: int = 42) -> gpd.GeoDataFrame:
    """
    Select points greedily while enforcing a minimum distance between all chosen points.
    Candidate order is randomized first so the final point pattern remains systematic-random rather than deterministic.
    """
    if points_gdf.empty:
        return points_gdf.iloc[0:0].copy()

    rng = random.Random(seed)
    idxs = list(points_gdf.index)
    rng.shuffle(idxs)

    selected = []
    for idx in idxs:
        pt = points_gdf.loc[idx].geometry
        if all(pt.distance(sel_pt) >= min_sep_m for sel_pt in selected):
            selected.append(pt)
        if len(selected) >= target_n:
            break

    out = points_gdf.loc[[i for i in idxs if points_gdf.loc[i].geometry in selected]].copy()
    out = out.iloc[:len(selected)].copy()
    return out


def sample_single_block(
    row: pd.Series,
    grid_size_m: float = 100,
    min_sep_m: float = 200,
    seed: int = 42,
) -> tuple[gpd.GeoDataFrame, gpd.GeoDataFrame]:
    """
    Generate the shifted 100 m grid and select valid plot centers for one block.
    The function returns both the selected points and the block-specific grid used to generate candidate locations.
    """
    block_geom = row["geometry"]

    grid_gdf, candidates = candidate_points_from_block(
        block_geom,
        grid_size_m=grid_size_m,
        seed=seed,
    )

    selected = select_points_with_min_spacing(
        candidates,
        target_n=int(row["target_n"]),
        min_sep_m=min_sep_m,
        seed=seed,
    )

    grid_gdf = grid_gdf.copy()
    grid_gdf["Block"] = row["id"]
    grid_gdf["Year"] = row["year"]
    grid_gdf["Stratum"] = row["Stratum"]
    grid_gdf["target_n"] = int(row["target_n"])

    if selected.empty:
        selected = gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=grid_gdf.crs)
        return selected, grid_gdf

    selected = selected.copy()
    selected["Block"] = row["id"]
    selected["Year"] = row["year"]
    selected["Stratum"] = row["Stratum"]
    selected["target_n"] = int(row["target_n"])
    selected["selected_n"] = len(selected)
    selected["constraint_limited"] = len(selected) < int(row["target_n"])

    return selected, grid_gdf


def sample_all_blocks(
    gdf: gpd.GeoDataFrame,
    grid_size_m: float = 100,
    min_sep_m: float = 200,
    seed: int = 42,
) -> tuple[gpd.GeoDataFrame, gpd.GeoDataFrame]:
    """
    Run the sampling workflow across all blocks and combine both selected points and block grids.
    A unique plot ID is assigned to the merged point layer and all per-block grids are merged for QA export.
    """
    all_pts = []
    all_grids = []

    for i, (_, row) in enumerate(gdf.iterrows()):
        pts, grid = sample_single_block(
            row,
            grid_size_m=grid_size_m,
            min_sep_m=min_sep_m,
            seed=seed + i,
        )

        if grid is not None and not grid.empty:
            all_grids.append(grid)

        if pts is not None and not pts.empty:
            all_pts.append(pts)

    if all_pts:
        points_out = pd.concat(all_pts, ignore_index=True)
        points_out = gpd.GeoDataFrame(points_out, geometry="geometry", crs=gdf.crs)
        points_out["Plot_ID"] = [f"P{i+1:04d}" for i in range(len(points_out))]
    else:
        points_out = gpd.GeoDataFrame(geometry=[], crs=gdf.crs)

    if all_grids:
        grid_out = pd.concat(all_grids, ignore_index=True)
        grid_out = gpd.GeoDataFrame(grid_out, geometry="geometry", crs=gdf.crs)
    else:
        grid_out = gpd.GeoDataFrame(geometry=[], crs=gdf.crs)

    return points_out, grid_out


def add_utm_and_latlon_fields(points_gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Add Easting and Northing in UTM 37S plus Latitude and Longitude in WGS84.
    This creates one output table that works for GPS, Excel reporting, and KML export.
    """
    out = points_gdf.copy()
    out["Easting_UTM37S"] = out.geometry.x
    out["Northing_UTM37S"] = out.geometry.y

    ll = out.to_crs("EPSG:4326")
    out["Longitude"] = ll.geometry.x
    out["Latitude"] = ll.geometry.y
    return out


def build_plot_footprints(points_gdf: gpd.GeoDataFrame, plot_size_m: float = 20) -> gpd.GeoDataFrame:
    """
    Create square 20 x 20 m plot footprints centered on the selected sampling points.
    These polygons are useful for QA in QGIS and for confirming block-edge clearance visually.
    """
    half = plot_size_m / 2.0
    polys = points_gdf.copy()
    polys["geometry"] = polys.geometry.apply(lambda p: box(p.x - half, p.y - half, p.x + half, p.y + half))
    return polys


def summarize_block_areas(gdf: gpd.GeoDataFrame) -> pd.DataFrame:
    """
    Create a clean block summary table with block ID, year, stratum, and recalculated hectares.
    This table is intended for reporting and proportional allocation documentation.
    """
    cols = ["id", "year", "Stratum", "area_ha_calc"]
    out = gdf[cols].copy()
    out = out.rename(columns={"id": "Block", "year": "Year", "area_ha_calc": "Area_ha"})
    return out.sort_values(["Stratum", "Year", "Block"]).reset_index(drop=True)


def export_sampling_outputs(
    points_gdf: gpd.GeoDataFrame,
    plot_polys_gdf: gpd.GeoDataFrame,
    grid_gdf: gpd.GeoDataFrame,
    block_summary_df: pd.DataFrame,
    out_dir: str | Path,
) -> None:
    """
    Export final sampling points, plot footprints, block grids, and block summaries to CSV and GeoPackage files.
    This version preserves the 100 m x 100 m systematic-random grid as a separate layer for GIS review and QA.
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    point_cols = [
        "Plot_ID",
        "Block",
        "Stratum",
        "Year",
        "Easting_UTM37S",
        "Northing_UTM37S",
        "Latitude",
        "Longitude",
        "geometry",
    ]

    points_gdf[point_cols].to_file(out_dir / "sampling_points.gpkg",layer="sampling_points",driver="GPKG")

    plot_polys_gdf.to_file(out_dir / "plot_footprints.gpkg",layer="plot_footprints",driver="GPKG")

    grid_gdf.to_file(out_dir / "sampling_grid_100m.gpkg",layer="sampling_grid_100m",driver="GPKG")

    points_gdf.drop(columns="geometry")[point_cols[:-1]].to_csv(out_dir / "sampling_points.csv",index=False)

    block_summary_df.to_csv(out_dir / "block_area_summary.csv", index=False)

In [76]:
blocks = load_blocks_geojson(blocks_path, work_crs=WORK_CRS)
blocks = assign_restoration_strata(blocks)
blocks = recalculate_block_area_ha(blocks)

corridor = load_corridor_geojson(corridor_path, work_crs=WORK_CRS)
corridor = recalculate_block_area_ha(corridor)
all_polys = pd.concat([blocks, corridor], ignore_index=True)
all_polys = gpd.GeoDataFrame(all_polys, geometry="geometry", crs=WORK_CRS)

In [77]:
sampling_zones = build_sampling_zone(all_polys, edge_buffer_m=EDGE_BUFFER_M)
sampling_zones = compute_target_plot_count(sampling_zones, sampling_pct=SAMPLING_PCT)

In [78]:
sample_points, sample_grid = sample_all_blocks(
    sampling_zones,
    grid_size_m=GRID_SIZE_M,
    min_sep_m=MIN_SEPARATION_M,
    seed=RANDOM_SEED,
)

sample_points = add_utm_and_latlon_fields(sample_points)
plot_footprints = build_plot_footprints(sample_points, plot_size_m=PLOT_SIZE_M)
block_summary = summarize_block_areas(all_polys)

In [82]:
export_sampling_outputs(
    points_gdf=sample_points,
    plot_polys_gdf=plot_footprints,
    grid_gdf=sample_grid,
    block_summary_df=block_summary,
    out_dir=out_dir,
)